# Intro to NN & PyTorch — Part 2: Building Networks (Lessons 6–7)

> 📚 **Part of a 3-notebook set** (Codecademy *Intro to NN with PyTorch*): **Part 1 — Foundations** · **Part 2 — Building Networks** · **Part 3 — Training**. Each notebook is self-contained (run its Setup cell first).

`nn.Sequential` → building a network **class** (subclassing `nn.Module`) → Python OOP reference.

*Continues from Part 1 (Foundations).*

In [1]:
# Setup — run once per session
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


torch version: 2.12.1+cu130
CUDA available: False


/home/plewis/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Lesson 6 — Build a Sequential Neural Network

Now we build a real network in PyTorch using the **`nn.Sequential`** container. Target architecture:

- **input layer:** 2 nodes
- **one hidden layer:** 3 nodes with **ReLU** activation
- **output layer:** 1 node (linear, no activation)

![Sequential network — input → hidden ReLU layer → linear output](images/sequential_network.png)

### Two building blocks (used inside `nn.Sequential`)
- **`nn.Linear(in, out)`** — defines a layer + does the linear calc (**weighted inputs + bias**). `in` = nodes feeding in, `out` = nodes it produces.
- **`nn.ReLU()`** — applies the ReLU activation to whatever came before it.

```python
model = nn.Sequential(
    nn.Linear(2, 3),   # 2 input nodes  -> 3 hidden nodes (weighted sum + bias)
    nn.ReLU(),         # ReLU on the 3 hidden nodes
    nn.Linear(3, 1)    # 3 hidden nodes -> 1 output node (linear)
)
```

- ⚠️ **Layers must align:** if a `nn.Linear(2, 3)` outputs 3, the **next** `nn.Linear` must start at **3** → `nn.Linear(3, 1)`.
- Weights/biases start **randomly initialized**; training (later) updates them to improve predictions.

> **Q:** *What is a `nn.Sequential` container?*
> **A:** A box you put layers into, in order; it pipes data **front to back**, feeding each module's output into the next. `model(X)` ≈ `Linear(2,3) → ReLU → Linear(3,1)` run in sequence — no branching, no skipping. It's itself an `nn.Module`, so it bundles all the layers' weights/biases into **one trainable object** (handy for handing to an optimizer later).
> - It's **one of two** ways to build a net: (1) `nn.Sequential` for a straight stack of layers (this lesson); (2) **subclass `nn.Module`** and write your own `forward()` when you need branches, skip connections, or conditional logic. Sequential is the beginner-friendly shortcut for the common case.

### Running feedforward
Feedforward = the first step of both **predicting** and **training**. To run it, pass an **input tensor** to the model: `model(X)`.

Input tensors are typically **2-D**:
- **rows** = individual examples (here, each row = one apartment). *Statisticians call these **observations**.*
- **columns** = features (here, e.g. building age, # bedrooms).

> **Naming convention:** the input tensor is named **`X`** (capital) — from math, signaling it's a **2-D matrix** (rows × columns), vs. lowercase `x` for a single value/vector.

```python
apts = np.array([[100, 3],   # 100 yrs old, 3 bedrooms
                 [50,  4]])  # 50 yrs old,  4 bedrooms
X = torch.tensor(apts, dtype=torch.float)
model(X)
# tensor([[-23.0715],
#         [-11.8710]], grad_fn=<AddmmBackward0>)
```

- Each **output row maps to the same input row**: `[100,3] → -23.07`, `[50,4] → -11.87`.
- Predictions are nonsense (negative rent!) because the model is **untrained** — expected.
- **`grad_fn=<AddmmBackward0>`** = PyTorch tracking the op for gradients; matters once we do **backpropagation**.

> **Q:** *Why does my `model(X)` output differ from the course's `-23.07 / -11.87`?*
> **A:** `nn.Linear` initializes weights/biases **randomly**, so untrained outputs differ every run unless you set a seed (`torch.manual_seed(...)`). The exact numbers don't matter pre-training — only the **shape** (2 inputs → 2 outputs) does. *(Our code cell seeds with `42` and happens to reproduce the course's `-23.0715 / -11.8710` exactly — so the course used the same seed.)*


In [2]:
# Build the Sequential model: 2 inputs -> 3 hidden (ReLU) -> 1 output
import numpy as np

torch.manual_seed(42)  # seed so the "random" init is reproducible across runs

model = nn.Sequential(
    nn.Linear(2, 3),   # 2 input nodes  -> 3 hidden nodes
    nn.ReLU(),         # ReLU on the hidden layer
    nn.Linear(3, 1)    # 3 hidden nodes -> 1 output node
)
print(model)

# --- run feedforward on two apartments: [age, bedrooms] ---
apts = np.array([[100, 3],   # 100 yrs old, 3 bedrooms
                 [50,  4]])  # 50 yrs old,  4 bedrooms
X = torch.tensor(apts, dtype=torch.float)

predictions = model(X)
print("\nfeedforward output (untrained):")
print(predictions)


Sequential(
  (0): Linear(in_features=2, out_features=3, bias=True)
  (1): ReLU()
  (2): Linear(in_features=3, out_features=1, bias=True)
)

feedforward output (untrained):
tensor([[-23.0715],
        [-11.8710]], grad_fn=<AddmmBackward0>)


### Exercise — Building Sequential models (checkpoints 1–3)

1. Net: input **3** → hidden **8** (ReLU) → output **1**. Assign to `model`.
2. Same, but add a **second hidden layer**: 4 nodes with **`nn.Sigmoid()`** activation.
3. Import StreeteEasy data (`size_sqft`, `bedrooms`, `building_age_yrs`) → tensor `X`. A model is pre-built with a **common "halving" architecture** (`3 → 16 → 8 → 4 → 1`, ReLU between each). Run the feedforward → `predicted_rent`.

**Notes:**
- Each checkpoint sets **`torch.manual_seed(42)`** (don't change it) so weights/biases init identically every run — reproducible outputs for the grader.
- **Halving architecture**: each hidden layer is ~half the size of the previous one as you feed toward the output (`16 → 8 → 4`). A common, sensible default shape.
- Activations can be **mixed/stacked** between linear layers (ReLU here, Sigmoid there) — `nn.Sequential` just applies whatever you list, in order.

> *Faking `streeteasy.csv`: I use the **exact 5 apartments the course previews** (`X[:5]`) as the stand-in DataFrame. Combined with `seed 42` + the same architecture, the feedforward reproduces the course's `predicted_rent[:5]` **exactly** (`-6.9229, -29.8163, …`). Real course line: `apartments_df = pd.read_csv("streeteasy.csv")`.*


In [3]:
# Checkpoint 1 — net: 3 inputs -> 8 hidden (ReLU) -> 1 output
torch.manual_seed(42)  # do not modify

model = nn.Sequential(
    nn.Linear(3, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

model


Sequential(
  (0): Linear(in_features=3, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

In [4]:
# Checkpoint 2 — add a 2nd hidden layer: 4 nodes with Sigmoid -> output
torch.manual_seed(42)  # do not modify

model = nn.Sequential(
    nn.Linear(3, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1)
)

model


Sequential(
  (0): Linear(in_features=3, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): Sigmoid()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

In [5]:
# Checkpoint 3 — feedforward real data through a "halving" net (3 -> 16 -> 8 -> 4 -> 1)
import pandas as pd

# --- Dataset import ---
# Real course code:  apartments_df = pd.read_csv("streeteasy.csv")
# FAKE stand-in = the exact 5 apartments the course previews, so outputs reproduce exactly:
apartments_df = pd.DataFrame({
    "size_sqft":        [480, 2000, 1000, 916, 975],
    "bedrooms":         [0,   2,    3,    1,   1],
    "building_age_yrs": [17,  96,   106,  29,  31],
})
apartments_numpy = apartments_df[['size_sqft', 'bedrooms', 'building_age_yrs']].values
X = torch.tensor(apartments_numpy, dtype=torch.float32)

# --- pre-built model (each hidden layer halves toward the output) ---
torch.manual_seed(42)  # do not modify
model = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

# --- feedforward ---
predicted_rent = model(X)

# preview first five predicted rents (untrained -> garbage, as expected)
predicted_rent[:5]


tensor([[ -6.9229],
        [-29.8163],
        [-16.0748],
        [-13.2427],
        [-14.1096]], grad_fn=<SliceBackward0>)

## Reference — Python OOP (for `nn.Module` subclassing)

Quick refresher, since building custom PyTorch models means subclassing `nn.Module`.

| Term | Meaning |
|---|---|
| **class** | a blueprint/template for objects |
| **instance** | a specific object built from a class |
| **`self`** | the current instance; auto-passed as the 1st arg of every method (`d.bark()` → `bark(d)`) |
| **attribute** | data stored on the object (`self.name`) |
| **method** | a function defined on the object |
| **inheritance** | a class reusing/extending another (`class Puppy(Dog)`) |
| **`super()`** | "call the parent class" (e.g. run the parent's `__init__`) |

```python
class Dog:
    species = "Canis"               # class attribute (shared)
    def __init__(self, name):       # constructor — runs on Dog("Rex")
        self.name = name            # instance attribute (per-object)
    def bark(self):
        return f"{self.name} woofs"
```

### `__init__` vs `__main__` (commonly confused — neither is a function you call)
- **`__init__`** — the **constructor method**. Python runs it *automatically* when you create an instance, to set up its attributes. You never call `__init__()` yourself.
- **`if __name__ == "__main__":`** — a **script guard**, not a method. `__name__` is `"__main__"` when the file is **run directly**, but is the module name when **imported** → so the guarded block runs only on direct execution, and is skipped on import.

```python
def main():
    ...
if __name__ == "__main__":   # runs only when this file is the script, not on import
    main()
```

### Why this matters here: the `nn.Module` pattern
Every custom model **subclasses `nn.Module`**, calls **`super().__init__()`** (sets up PyTorch's internals — required), defines layers as attributes in `__init__`, and implements **`forward()`**:

```python
class Net(nn.Module):
    def __init__(self):
        super().__init__()              # MUST call first
        self.hidden = nn.Linear(2, 3)
        self.output = nn.Linear(3, 1)
    def forward(self, x):               # defines the feedforward path
        x = torch.relu(self.hidden(x))
        return self.output(x)
```

> **Q:** *What is `super()`?*
> **A:** In one line: **`super()` = the parent class; `super().__init__()` = "let the parent set itself up before I do mine."** It's plain Python (not PyTorch). For `class Net(nn.Module)`, the parent is `nn.Module`, so `super().__init__()` runs `nn.Module.__init__(self)`, which sets up PyTorch's internal bookkeeping (the dicts that track your parameters & submodules). **Skip it → those dicts never exist → `self.layer = nn.Linear(...)` errors or `.parameters()` comes back empty and training does nothing.** Rule of thumb: **the first line of every `nn.Module.__init__` is `super().__init__()`.** Same mechanic as `Puppy` calling `Dog`'s `__init__` above — just with `nn.Module` as the parent.
>
> **⚠️ Precise wording:** `super()` doesn't *define* the constructor — it **calls** one that already exists. `nn.Module` already has its own `__init__` (written by PyTorch). When you write your **own** `__init__` in `Net`, you **override** it → Python now runs *yours instead of* the parent's, so the parent's no longer fires automatically. `super().__init__()` is how you **manually run the parent's already-defined `__init__`** before adding your own setup. *(Recipe analogy: the parent's `__init__` is a recipe already written; `super().__init__()` doesn't write it — it **cooks it** before you add your own ingredients.)*

> **`nn.Sequential` vs subclassing:** Sequential = the shortcut for a straight stack of layers. Subclassing `nn.Module` = full control of `forward()` (branches, skip connections, conditionals). The cell below shows the **same network both ways** — identical output.


In [6]:
# Same 2 -> 3 (ReLU) -> 1 network, built two ways -> identical output

# Way 1: nn.Sequential (the shortcut)
torch.manual_seed(42)
seq_model = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Linear(3, 1))

# Way 2: subclass nn.Module (full control of forward())
class Net(nn.Module):
    def __init__(self):
        super().__init__()                 # required: set up nn.Module internals
        self.hidden = nn.Linear(2, 3)
        self.output = nn.Linear(3, 1)
    def forward(self, x):
        x = torch.relu(self.hidden(x))     # same ReLU between the two Linear layers
        return self.output(x)

torch.manual_seed(42)                      # same seed -> same random init
class_model = Net()

X = torch.tensor([[100.0, 3.0], [50.0, 4.0]])
print("Sequential:", seq_model(X).flatten().tolist())
print("Subclassed:", class_model(X).flatten().tolist())
print("identical? ", torch.allclose(seq_model(X), class_model(X)))


Sequential: [-23.071517944335938, -11.871048927307129]
Subclassed: [-23.071517944335938, -11.871048927307129]
identical?  True


## Lesson 7 — Build a Neural Network Class

`nn.Sequential` is handy, but it can only feed data **straight from one layer to the next**. Real-world nets often need **non-sequential** logic — skipping layers, looping, branching — and that flexibility comes from building the network with **OOP** (subclassing `nn.Module`). Goal here: navigate OOP NN code, not master OOP. (See the [OOP reference appendix](#Reference-—-Python-OOP-(for-nn.Module-subclassing)) for the underlying Python.)

We'll rebuild the **same `3 → 16 → 8 → 4 → 1` halving net** from the Sequential exercise — this time as a class.

### 1. Create the class
`nn.Sequential` is a *class* (a **type** of network); `model = nn.Sequential(...)` makes an **instance**. We can define our own type by subclassing `nn.Module`:

```python
class NN_Regression(nn.Module):
```

### 2. Initialize the components — `__init__` ("gather the ingredients")
Inside the class, set up every layer + activation you'll use, as `self.` attributes:

```python
def __init__(self):
    super(NN_Regression, self).__init__()   # run nn.Module's setup first (required)
    # layers
    self.layer1 = nn.Linear(3, 16)
    self.layer2 = nn.Linear(16, 8)
    self.layer3 = nn.Linear(8, 4)
    self.layer4 = nn.Linear(4, 1)
    # activation
    self.relu = nn.ReLU()
```

- **`self.`** = store it on the object so the next method (`forward`) can reuse it (`self.layer1`, `self.relu`).
- `super(NN_Regression, self).__init__()` is the **older explicit form** of `super().__init__()` — same effect (the modern shorthand works too).

### 3. Define the forward pass — `forward` ("combine the ingredients")
Describe how an input tensor `x` flows layer → layer:

```python
def forward(self, x):
    x = self.layer1(x)
    x = self.relu(x)
    x = self.layer2(x)
    x = self.relu(x)
    x = self.layer3(x)
    x = self.relu(x)
    x = self.layer4(x)   # no activation on the final layer
    return x
```

- `x = self.layer1(x)` passes `x` through that layer and reassigns the result; chaining these *is* the feedforward.
- This is where OOP earns its keep: you could add `if`/loops/skip-connections here — impossible in `nn.Sequential`.
- **You don't call `forward()` directly** — calling the instance, `model(X)`, invokes it for you (PyTorch routes `model(X)` → `model.forward(X)`).

### 4. Instantiate
The class is just a *type*. Make an actual network with:

```python
model = NN_Regression()
```

> **Sequential vs class — same net:** because it's the same `3→16→8→4→1` architecture with the same `seed 42`, this class produces the **exact same `predicted_rent`** as Checkpoint 3 (`-6.9229, …`). OOP didn't change the math here — it just gave us a `forward()` we *could* customize.


In [7]:
# Lesson 7 — the halving net (3->16->8->4->1) built as an OOP class
import pandas as pd

class NN_Regression(nn.Module):
    def __init__(self):
        super(NN_Regression, self).__init__()   # run nn.Module's setup (required)
        # 1. gather ingredients: layers + activation
        self.layer1 = nn.Linear(3, 16)
        self.layer2 = nn.Linear(16, 8)
        self.layer3 = nn.Linear(8, 4)
        self.layer4 = nn.Linear(4, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # 2. combine ingredients: define the feedforward path
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))
        x = self.layer4(x)            # no activation on the output layer
        return x

# instantiate (seed 42 so init matches Checkpoint 3's Sequential net)
torch.manual_seed(42)
model = NN_Regression()
print(model)

# same faked StreeteEasy data as Checkpoint 3
apartments_df = pd.DataFrame({
    "size_sqft":        [480, 2000, 1000, 916, 975],
    "bedrooms":         [0,   2,    3,    1,   1],
    "building_age_yrs": [17,  96,   106,  29,  31],
})
X = torch.tensor(apartments_df.values, dtype=torch.float32)

# model(X) auto-calls model.forward(X)
predicted_rent = model(X)
print("\npredicted_rent[:5] (identical to Checkpoint 3's Sequential net):")
print(predicted_rent[:5])


NN_Regression(
  (layer1): Linear(in_features=3, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (layer3): Linear(in_features=8, out_features=4, bias=True)
  (layer4): Linear(in_features=4, out_features=1, bias=True)
  (relu): ReLU()
)

predicted_rent[:5] (identical to Checkpoint 3's Sequential net):
tensor([[ -6.9229],
        [-29.8163],
        [-16.0748],
        [-13.2427],
        [-14.1096]], grad_fn=<SliceBackward0>)


### Exercise — Neural network classes (checkpoints 1–3)

1. **`NN_Regression`** (the `3→16→8→4→1` net) + feedforward → output matches the Sequential net from before (`-6.9229, …`). *This is exactly the class cell above, so it's not repeated here.*
2. **`OneHidden`** — a small net (input **2** → hidden **4** → output **1**). The `forward` is left blank; filling it in (`layer1 → relu → layer2`) is the task. With an **empty** `forward` that just `return x`, the input passes through **unchanged** (no layers applied). Input `[3, 4.5]` → output `2.4422`.
3. **Parameterized hidden size** — same `OneHidden`, but `__init__` takes a **`numHiddenNodes`** argument so the hidden layer width is set at instantiation: `nn.Linear(2, numHiddenNodes)` → `nn.Linear(numHiddenNodes, 1)`. `OneHidden(10)` → `1.2633`.

**Why this is the payoff of OOP:**
- A **parameter on `__init__`** lets one class spin up many architectures — `OneHidden(4)`, `OneHidden(10)`, `OneHidden(64)` — without rewriting the class. (And `OneHidden(4)` reproduces Checkpoint 2's `2.4422` exactly, since it's the same net.)
- Changing the hidden width **changes the output** (`4` → `2.4422`, `10` → `1.2633`): more nodes = more (randomly-initialized) weights, so untrained predictions differ. Training will later tune whichever size you pick.

> **Note:** these use a hand-built input tensor `[3, 4.5]` (not StreeteEasy), so no faked CSV needed — they run as-is and match the course exactly.


In [8]:
# Checkpoint 2 — OneHidden (2 -> 4 -> 1); fill in the forward pass
torch.manual_seed(42)  # do not modify

class OneHidden(nn.Module):
    def __init__(self):
        super(OneHidden, self).__init__()
        self.layer1 = nn.Linear(2, 4)
        self.layer2 = nn.Linear(4, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # SOLUTION: x flows layer1 -> relu -> layer2
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x
        # (If forward just did `return x`, the input would pass through UNCHANGED.)

model = OneHidden()

X = torch.tensor([3, 4.5], dtype=torch.float32)
predictions = model(X)
predictions                                    # -> 2.4422


tensor([2.4422], grad_fn=<ViewBackward0>)

In [9]:
# Checkpoint 3 — OneHidden with a configurable hidden-layer width (numHiddenNodes)
torch.manual_seed(42)  # do not modify

class OneHidden(nn.Module):
    def __init__(self, numHiddenNodes):       # hidden width now a constructor arg
        super(OneHidden, self).__init__()
        self.layer1 = nn.Linear(2, numHiddenNodes)   # 2 inputs -> variable hidden
        self.layer2 = nn.Linear(numHiddenNodes, 1)   # variable hidden -> 1 output
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = OneHidden(10)                          # 10 hidden nodes (try 4 -> matches CP2)

input_tensor = torch.tensor([3, 4.5], dtype=torch.float32)
predictions = model(input_tensor)
predictions                                    # 10 nodes -> 1.2633


tensor([1.2633], grad_fn=<ViewBackward0>)